# BorrowBox V2.1 Coding Agent Prompt — Community + Ownership Foundation

Use the repository's V2 planning notebooks as the source of truth before modifying code:

```text
docs/BORROWBOX_V2_INDEX.ipynb
docs/BORROWBOX_V2_VISION.ipynb
docs/BORROWBOX_PRODUCT_SPEC.ipynb
docs/BORROWBOX_ARCHITECTURE.ipynb
docs/BORROWBOX_TRANSACTION_SPEC.ipynb
docs/BORROWBOX_ROADMAP.ipynb
docs/BORROWBOX_DECISIONS.ipynb
docs/BORROWBOX_V2_1_DATABASE_SCHEMA.ipynb
docs/BORROWBOX_V2_1_SEED_DATA_AND_INITIALIZATION.ipynb
```

## Objective

Implement **BorrowBox V2.1 — Community + Ownership Foundation** as a gradual evolution of the existing codebase.

The goal is to make BorrowBox genuinely community-based and establish the correct ownership model before implementing V2.2 transactions.

## Critical boundary

**V2.1 starts from a FRESH DATABASE.**

Do NOT build a V1→V2 migration layer.

Do NOT:

```text
migrate V1 Groups into V2 Communities
migrate V1 Items into V2 Assets
migrate V1 BorrowRequests into Transactions
read V1 legacy tables at runtime
add compatibility repositories solely for V1
```

The V1 code/history remains preserved in Git. V2.1 has its own schema and deterministic seed data.

## Domain model

Implement the V2.1 model:

```text
User
  ↓
Membership
  ↓
Community

User
  ↓
Asset
  ↓
AssetUnit
  ↓
CommunityListing
```

### Hard invariants

1. A User is not permanently a borrower or lender.
2. Membership is a first-class entity.
3. Community-specific role/context belongs to Membership.
4. Membership context uses MySQL `JSON` as `context_metadata`.
5. An Asset is owned independently of communities.
6. One Asset may be listed in multiple Communities.
7. A CommunityListing requires the Asset owner to have ACTIVE Membership in the target Community.
8. Asset has NO `total_quantity` field.
9. AssetUnits are materialized immediately.
10. Creating an Asset with N units must insert the Asset and exactly N AssetUnits in one atomic database transaction.
11. Quantity and availability derive from AssetUnits.
12. A CommunityListing has no independent quantity.
13. Public Explore/search responses aggregate AssetUnits into one Asset summary per CommunityListing.
14. Public Explore/search responses do not expose AssetUnit IDs.
15. Backend/database is authoritative for availability.
16. Leave concurrency/reservation implementation for V2.2, but do not design V2.1 in a way that blocks safe locking later.
17. The same creator/manager cannot have two ACTIVE communities with the same name.
18. V2.1 uses deterministic seed/reset data.
19. No V1 migration layer is implemented.

## Scope

### Implement

```text
Community
Membership
Community roles
Membership context metadata
Community creation
Community joining/leaving according to the chosen V2.1 rules
Basic manager membership controls
Community type
Community location fields/model as specified in the architecture
Community rules foundation
Asset
AssetUnit
CommunityListing
Selective community visibility
Community-scoped Explore
Community-scoped Inventory
Repeatable development seed/reset
```

### Do not implement

```text
Transaction
Transaction messaging
Handover
Four-photo evidence
Loan timer
Extensions
Reputation
Badges
Flags
AI
Tribunal
```

Design compatibility with these future features, but do not prematurely implement them.

## Seed dataset

Use `docs/BORROWBOX_V2_1_SEED_DATA_AND_INITIALIZATION.ipynb` as the required baseline.

The seed data must include:

```text
3 communities
multiple memberships
community managers
community-specific context metadata
an Asset with 2 physical units
the same Asset listed in 3 communities
single-unit assets
at least one unlisted Asset
```

The Football example is mandatory because it tests cross-community shared availability.

## API behavior

Public Explore/search should return an aggregate summary such as:

```text
assetId
title
description
category
listingStatus
totalUnits
availableUnits
borrowedUnits
```

Do not expose AssetUnit IDs.

The owner/manager UI may later expose physical units for advanced workflows, but public V2.1 search does not.

## Authorization

For listing creation/update:

```text
Asset.owner_id
    ↓
ACTIVE Membership
    ↓
same Community
```

must be verified server-side.

Do not rely only on frontend controls.

## V1 code preservation

Do not delete working V1 functionality merely because V2 uses a fresh database.

Reuse existing authentication infrastructure where compatible.

Refactor only where required by the V2.1 domain model.

## Database

Use the project's existing MySQL 8.0 setup.

Before adding tables/entities, inspect existing code/configuration and choose the simplest maintainable schema initialization strategy.

Do not introduce a migration framework solely for the sake of migration.

Because V2 starts with a fresh database, the schema can be created cleanly for V2.1.

## Seed/reset

The development environment must be able to recreate the same logical baseline repeatedly.

A clean reset should result in:

```text
fresh database
→ V2.1 schema
→ deterministic seed users
→ deterministic seed communities
→ deterministic memberships
→ deterministic assets
→ deterministic AssetUnits
→ deterministic CommunityListings
```

No duplicate seed rows should appear.

## Resolved safety requirements

### MySQL JSON

`Membership.context_metadata` is MySQL `JSON`.

Prefer Hibernate native JSON mapping:

```java
@JdbcTypeCode(SqlTypes.JSON)
private Map<String, Object> contextMetadata;
```

Do not add Hypersistence Utils unless native Hibernate mapping is proven insufficient.

Include an integration test proving JSON persist/read round-trip.

### Transactional AssetUnit creation

When Asset creation with quantity `N` is implemented:

- wrap Asset + N AssetUnit creation in one Spring-managed transaction;
- use `@Transactional` at the service/use-case boundary;
- do not require `rollbackFor = Exception.class` unless checked exceptions in the actual implementation require it;
- add an integration test that forces a failure during unit creation and verifies the Asset and all partial AssetUnits are rolled back.

### Community admission

There is no open-for-all mode.

Supported modes:

```text
MANAGER_APPROVAL
LOCATION_VERIFIED
```

For location verification, check the user's explicit one-time location against the Community's configured radius. Do not implement continuous tracking.

### Community active-name uniqueness

Enforce the locked normalized uniqueness rule at database level using the active-name key described in the schema.

### Schema/seed

Use:

```text
schema.sql
ddl-auto=validate
ApplicationRunner (idempotent seed)
borrowbox.seed.enabled=true
```

Do not introduce Flyway/Liquibase solely for V2.1 migration.
Do not add a V1 runtime migration layer.

## Testing

Before declaring V2.1 complete:

### Backend

Run:

```text
mvn test
```

Add tests for at least:

```text
community creation
membership uniqueness
membership role/context
asset creation with N AssetUnits
atomic Asset + AssetUnit creation
listing authorization
duplicate asset/community listing prevention
aggregate availability
```

### Frontend

Run:

```text
npm run build
```

Add/update Cypress coverage for:

```text
community list
active community selection
community-scoped Explore
community-scoped Inventory
asset listed in multiple communities
aggregate availability presentation
unlisted asset is absent from Explore
```

### Full E2E

Run:

```text
npx cypress run --headless
```

### Docker

Verify:

```text
docker compose ps
```

and manually inspect the application with the deterministic V2.1 seed data.

## Important workflow

Do not implement the entire milestone in one uncontrolled change.

Work in slices:

```text
V2.1.1 Community + Membership
→ test
→ review
→ commit

V2.1.2 Community creation/join/leave
→ test
→ review
→ commit

V2.1.3 Roles/context
→ test
→ review
→ commit

V2.1.4 Asset + AssetUnit
→ test
→ review
→ commit

V2.1.5 CommunityListing
→ test
→ review
→ commit

V2.1.6 Community-scoped Explore/Inventory
→ test
→ review
→ commit

V2.1.7 Seed/reset + final QA
→ test
→ review
→ commit
```

Before each slice, explain which existing files will change and why.

Do not make unrelated refactors.

## Completion condition

Do not report "V2.1 complete" merely because the backend compiles.

V2.1 is complete when the following real flow works:

```text
Create/join community
        ↓
Receive community role/context
        ↓
Create one owned asset with multiple physical units
        ↓
Choose multiple communities for that asset
        ↓
See one asset in those communities
        ↓
See shared aggregate availability
        ↓
Keep an unlisted asset private
        ↓
Reject unauthorized community listing attempts
        ↓
Reset database
        ↓
Reproduce the same seed dataset
```

At the end, provide:

```text
files changed
database schema changes
seed/reset mechanism
tests added
test results
manual QA steps
known limitations
git commit
```

Do not claim any feature is implemented unless it is actually verified.
